# AmSC IRI Job Submission (Networking Optional)

AmSC Resource Orchestration Toolkit (AmSCROT) - Orchestrating Infrastructure Service capabilities.

This notebook demonstrates how to submit compute jobs to **ESnet IRI East** and **ESnet IRI West** sites using the `amscrot` Client module, with optional SENSE network provisioning between the sites. 

## Workflow Overview

1. **Initialize** the AmSCROT client
2. **Create** a session
3. *(Optional)* **Provision** a SENSE L2VPN network between sites
4. **Set up** ESnet IRI service clients for East and West
5. **Discover** available compute resources at each site
6. **Define** and submit batch jobs
7. **Monitor** job status
8. **Clean up**

## Prerequisites

### Required Packages
- `amscrot-py` (installed)
- `python-dotenv` (installed)

### Installation

```
pip install amscrot-py python-dotenv
```

### Credentials

ESnet IRI service clients authenticate via API keys stored in `~/.amscrot/credentials.yml`. You need entries for **both** the East and West sites:

```yaml
# ~/.amscrot/credentials.yml

esnet-iri-east:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://iri-dev.ppg.es.net

esnet-iri-west:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://esnet-west.sdn-sense.net/
```

If you also want SENSE network provisioning, add a `sense` section, for example::

```yaml
sense:
  AUTH_ENDPOINT: https://sense-o.es.net:8543/auth/realms/StackV/protocol/openid-connect/token
  API_ENDPOINT: https://sense-o-dev.es.net:8443/StackV-web/restapi
  CLIENT_ID: Portal
  USERNAME: <your-username>
  PASSWORD: <your-password>
  SECRET: None
  verify: False
```

---
## 0. Retrieve an authentication token

The process of issuing AmSC tokens is evolving rapidly. For IRI API usage, see the following repository for examples on how to retrieve a token for use with this toolkit.

  * https://github.com/doe-iri/iri-facility-api-examples

Once you have a token, copy it to a `.env` file, set in the `AMSC_TOKEN` environment variable, or paste into the following cell to generate a new credential file for this toolkit example.

In [ ]:
import os
import yaml
from dotenv import load_dotenv

AMSC_TOKEN = None   # <-- manually set token
if not AMSC_TOKEN:
    load_dotenv()   # take environment variables from .env file (if present)
    AMSC_TOKEN = os.getenv("AMSC_TOKEN")

# Create credentials data
credentials = {
    'esnet-iri-east': {
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://iri-dev.ppg.es.net'
    },
    'esnet-iri-west': {
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://esnet-west.sdn-sense.net/'
    }
}

# Write to ~/.amscrot/credentials-new.yml
cred_path = os.path.expanduser("~/.amscrot/credentials-new.yml")
os.makedirs(os.path.dirname(cred_path), exist_ok=True)
with open(cred_path, 'w') as f:
    yaml.dump(credentials, f)
print(f"Wrote credentials to {cred_path}")

---
## 0.1 Use SENSE Auth Token (Optional) 

If you already have a SENSE-O credentials file, you can use it to generate a token that works with the ESnet IRI endpoints.

In [ ]:
import os
import yaml
from sense.client.apiclient import ApiClient
from sense.common import getConfig

# Force use of the new auth file
os.environ['SENSE_AUTH_OVERRIDE'] = os.path.expanduser("~/.sense-o-auth-new.yaml")

# Get config and token
config = getConfig()
sense_client = ApiClient(config)
token = sense_client.token['access_token']
print(f"Got SENSE token: {token[:10]}...")

# Create credentials data
credentials = {
    'sense': {
        'AUTH_ENDPOINT': config.get('AUTH_ENDPOINT'),
        'API_ENDPOINT': config.get('API_ENDPOINT'),
        'CLIENT_ID': config.get('CLIENT_ID'),
        'USERNAME': config.get('USERNAME'),
        'PASSWORD': config.get('PASSWORD'),
        'SECRET': config.get('SECRET'),
        'verify': config.get('verify', False)
    },
    'esnet-iri-east': {
        'api_key': token,
        'api_endpoint': 'https://iri-dev.ppg.es.net'
    },
    'esnet-iri-west': {
        'api_key': token,
        'api_endpoint': 'https://esnet-west.sdn-sense.net/'
    }
}

# Write to ~/.amscrot/credentials-new.yml
cred_path = os.path.expanduser("~/.amscrot/credentials-new.yml")
os.makedirs(os.path.dirname(cred_path), exist_ok=True)
with open(cred_path, 'w') as f:
    yaml.dump(credentials, f)
print(f"Wrote credentials to {cred_path}")

---
## 1. Initialize Client & Session

In [ ]:
import time
from amscrot.client.client import Client
from amscrot.client.job import Job, JobType, JobServiceType, JobSpec, JobState
from amscrot.serviceclient import ServiceClient
from amscrot.util.constants import Constants

client = Client()
session = client.create_session("sense-networked-jobs")
print("Client and session initialized.")

## 2. (Optional) Add SENSE Network

Set `USE_NETWORK = True` to provision a SENSE L2VPN between the East and West sites. This requires valid SENSE credentials in your credentials file.

In [ ]:
USE_NETWORK = False  # Set to True to enable SENSE network provisioning

if USE_NETWORK:
    sense_provider = client.add_provider(
        label="sense",
        type="sense",
        name="sense-provider",
        profile="sense",
        credential_file="~/.amscrot/credentials-new.yml"
    )
    net1 = session.add_network(
        label="net1",
        provider=sense_provider,
        name_prefix="test-net",
        site="ESnet",
        profile="AmSC-WFC-L2VPN",
        count=1
    )
    print("SENSE network added to session.")
else:
    print("Skipping SENSE network provisioning.")

## 3. Set Up ESnet IRI Service Clients

We create two service clients — one for each IRI site. Each client loads its credentials from the corresponding profile in `~/.amscrot/credentials-new.yml`.

In [ ]:
east_client = ServiceClient.create(
    type=Constants.ServiceType.AMSC_IRI,
    name="iri-east",
    profile="esnet-iri-east",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(east_client)

west_client = ServiceClient.create(
    type=Constants.ServiceType.AMSC_IRI,
    name="iri-west",
    profile="esnet-iri-west",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(west_client)

## 4. Discover Compute Resources

Each service client's `discover()` method returns a `DiscoveryResult` container with typed accessors for each resource type. We use `.compute` to find available compute resources at each site.

In [ ]:
east_discovery = east_client.discover()
west_discovery = west_client.discover()

print(f"East discovery: {east_discovery.summary()}")
print(f"West discovery: {west_discovery.summary()}")

assert east_discovery.compute, "No compute resources found on East site!"
assert west_discovery.compute, "No compute resources found on West site!"

east_resource_id = east_discovery.compute[0].data.get("id")
west_resource_id = west_discovery.compute[0].data.get("id")

print(f"\nEast resource_id: {east_resource_id}")
print(f"West resource_id: {west_resource_id}")

## 5. Define Job Specs & Jobs

Each job is a simple batch job that runs `/bin/echo` on the discovered compute resource. The `resource_id` is set dynamically from the discovery step above.

In [ ]:
common_resources = {
    "node_count": 1,
    "process_count": 1,
    "processes_per_node": 1,
    "cpu_cores_per_process": 1,
    "gpu_cores_per_process": 1,
    "exclusive_node_use": True,
    "memory": 268435456
}

common_attributes = {
    "directory": "/data/home/kissel",  # <-- adjust
    "duration": 60,
    "queue_name": "debug",
    "account": "interactive",
    "stdout_path": "/data/home/kissel/stdout.log",  # <-- adjust
    "stderr_path": "/data/home/kissel/stderr.log"
}

spec_east = JobSpec(
    executable="/bin/echo",
    arguments=["Hello AmSC East"],
    resources=common_resources,
    attributes={"resource_id": east_resource_id, **common_attributes}
)

spec_west = JobSpec(
    executable="/bin/echo",
    arguments=["Hello AmSC West"],
    resources=common_resources,
    attributes={"resource_id": west_resource_id, **common_attributes}
)

job1 = Job(name="job-1", type=JobType.COMPUTE, service_type=JobServiceType.BATCH,
           service_client=east_client, job_spec=spec_east)

job2 = Job(name="job-2", type=JobType.COMPUTE, service_type=JobServiceType.BATCH,
           service_client=west_client, job_spec=spec_west)

session.add_job(job1)
session.add_job(job2)

print("Jobs defined and added to session.")

## 6. Plan

The plan phase validates all resources and job specs before anything is created.

In [ ]:
session.plan()
session.show()

## 7. Apply and monitor

Apply creates resources (if networking is enabled) and submits the compute jobs.
`session.wait()` then polls both jobs until they complete (or raise `WaitTimeoutError`).

In [ ]:
try:
    session.apply()
    print("Session applied successfully.")
except Exception as e:
    print(f"Failed to apply session: {e}")
    raise

results = session.wait(
    jobs=[job1, job2],
    target_states=[JobState.COMPLETED, JobState.FAILED, JobState.CANCELED],
    timeout=120.0,
    interval=2.0,
    verbose=True,
)

s1 = results["job-1"]
s2 = results["job-2"]

assert s1.state == JobState.COMPLETED, f"Job1 failed or timed out: {s1}"
assert s2.state == JobState.COMPLETED, f"Job2 failed or timed out: {s2}"
print(f"\n✅ Both jobs completed successfully!")

## 8. View job logs

Fetch files from the Session and display stdout logs.

In [ ]:
fetched = session.fetch_output_files(jobs=[job1, job2])
print(f"Fetched files: {fetched}")

for job_name, paths in fetched.items():
    stdout_path = paths.get("stdout")
    if stdout_path:
        print(f"\n--- stdout for {job_name} ---")
        try:
            with open(stdout_path) as f:
                print(f.read())
        except Exception as e:
            print(f"  (could not read stdout: {e})")

## 9. Clean Up

Destroy the session to tear down any provisioned resources and cancel remaining jobs.

In [ ]:
session.destroy()